In [0]:
fact_remittance = dbutils.widgets.get("fact_remittance")
yrinvo = dbutils.widgets.get("yrinvo")
hchbofficemapping = dbutils.widgets.get("hchbofficemapping")
oblisthistory = dbutils.widgets.get("oblisthistory")
billing_reference = dbutils.widgets.get("billing_reference")
office = dbutils.widgets.get("office")
payerdimension = dbutils.widgets.get("payerdimension")
cubeserviceofficetxnsourcesystem = dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
date = dbutils.widgets.get("date")
client_episode_fs = dbutils.widgets.get("client_episode_fs")
client_episodes_all = dbutils.widgets.get("client_episodes_all")
client = dbutils.widgets.get("client")
denialcodemapping = dbutils.widgets.get("denialcodemapping")
denialtype = dbutils.widgets.get("denialtype")
abilityremittancedetails=dbutils.widgets.get("abilityremittancedetails")
mart_remittance=dbutils.widgets.get("mart_remittance")


In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {fact_remittance}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {fact_remittance} executed")
    spark.sql(f"""
    
INSERT INTO {fact_remittance}
(
    reporting_week_ending_date_key,
    create_date_key,
    payment_date_key,
    statement_start_date_key,
    statement_end_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    enterprise_category_key,
    denial_type_key,
    payer_icn,
    ability_payor_name,
    patient_name,
    member_id,
    adjustment_codes,
    remark_code,
    patientctlno,
    invoice_number,
    charge,
    total_service_adjustments,
    claim_payment,
    underpayment_amount,
    check_eftnumber,
    loaded_ts
)
SELECT
    CAST(Reporting_Week_Ending_Date_Key AS INT),
    CAST(Create_Date_Key AS INT),
    CAST(Payment_Date_Key AS INT),
    CAST(Statement_Start_Date_Key AS INT),
    CAST(Statement_End_Date_Key AS INT),
    CAST(Source_System_Key AS INT),
    CAST(Office_Key AS INT),
    CAST(Payor_Key AS INT),
    CAST(Client_Key AS INT),
    CAST(Enterprise_Category_Key AS INT),
    CAST(Denial_Type_Key AS INT),
    CAST(Payer_ICN AS STRING),
    CAST(Ability_Payor_Name AS STRING),
    CAST(Patient_Name AS STRING),
    CAST(Member_ID AS STRING),
    CAST(Adjustment_Codes AS STRING),
    CAST(Remark_Code AS STRING),
    CAST(PatientCtlNo AS STRING),
    CAST(Invoice_Number AS STRING),
    CAST(Charge AS DOUBLE),
    CAST(Total_Service_Adjustments AS DOUBLE),
    CAST(Claim_Payment AS DOUBLE),
    CAST(Underpayment_Amount AS DOUBLE),
    CAST(Check_EFTNumber AS STRING),
    current_timestamp() AS loaded_ts
FROM {mart_remittance}
    """)
 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp1_base AS
SELECT 
    BranchID,
    InvNum,
    GroupID,
    CltId,
    PrimID
FROM {yrinvo}
WHERE BranchID <> 'COR'
""")


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp1 AS
SELECT 
    CASE 
        WHEN t.BranchID RLIKE '[A-Z]' THEN om.TargetOfficeNumber
        ELSE t.BranchID
    END AS BranchID,
    t.InvNum,
    t.GroupID,
    t.CltId,
    t.PrimID
FROM tmp1_base t
LEFT JOIN {hchbofficemapping} om
    ON t.BranchID = om.SourceOfficeCode
""")


In [0]:

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp2 AS
SELECT office, invno, payorname, PAYORTYPE, BILLTO, CLIENTNO
FROM (
    SELECT
        office,
        invno,
        payorname,
        PAYORTYPE,
        BILLTO,
        CLIENTNO,
        ROW_NUMBER() OVER (PARTITION BY invno, office ORDER BY invno, office) AS rnb
    FROM {oblisthistory}
    WHERE invno <> 'adv'
) a
WHERE rnb = 1
""")


In [0]:
spark.sql(f"""
INSERT INTO {fact_remittance} 
(
    reporting_week_ending_date_key,
    create_date_key,
    payment_date_key,
    statement_start_date_key,
    statement_end_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    enterprise_category_key,
    denial_type_key,
    payer_icn,
    ability_payor_name,
    patient_name,
    member_id,
    adjustment_codes,
    remark_code,
    patientctlno,
    invoice_number,
    charge,
    total_service_adjustments,
    claim_payment,
    underpayment_amount,
    check_eftnumber,
    loaded_ts
)
-- First SELECT - HCHB System (BHHC grouping with specific patterns)
SELECT 

    CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS create_date_key,
    dt2.DateKey AS payment_date_key,
    dt3.DateKey AS statement_start_date_key,
    dt4.DateKey AS statement_end_date_key,
    s.SourceSystemKey AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    b.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.PayerICN AS payer_icn,
    ard.PayerName AS ability_payor_name,
    ard.PatientName AS patient_name,
    ard.MemberID AS member_id,
    ard.AdjustmentCodes AS adjustment_codes,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS  patientctlno,
    ard.PatientCtlNo AS invoice_number,
    CAST(ard.Charge AS DECIMAL(19,4)) AS charge,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS total_service_adjustments,
    CAST(ard.ClaimPayment AS DECIMAL(19,4)) AS claim_payment,
    CAST(ard.UnderPaymentAmount AS DECIMAL(19,4)) AS underpayment_amount,
    ard.Check_EFTNumber AS check_eftnumber,
    CURRENT_TIMESTAMP() AS Loaded_ts
FROM {abilityremittancedetails} ard

JOIN tmp1 inv
    ON CAST(ard.PatientCtlNo AS STRING) = CAST(inv.InvNum AS STRING)
    AND (ard.PatientCtlNo RLIKE '^2[89][90123].*' OR ard.PatientCtlNo RLIKE '^1.*')

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ABS(CAST(inv.BranchID AS INT))

LEFT JOIN {payerdimension} pd
    ON CAST(inv.GroupID AS STRING) = CAST(pd.PayerID AS STRING)

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'HCHB'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = CAST(ard.CreateDate AS DATE)

LEFT JOIN {date} dt2
    ON dt2.CalendarDate = CAST(ard.PaymentDate AS DATE)

LEFT JOIN {date} dt3
    ON dt3.CalendarDate = CAST(ard.StatementStart AS DATE)

LEFT JOIN {date} dt4
    ON dt4.CalendarDate = CAST(ard.StatementEnd AS DATE)

LEFT JOIN {client_episode_fs} cefs
    ON CAST(inv.PrimID AS BIGINT) = CAST(cefs.cefs_id AS BIGINT)

LEFT JOIN {client_episodes_all} cea
    ON cea.epi_id = cefs.cefs_epiid

LEFT JOIN (
    SELECT *
    FROM (
        SELECT 
            clientKey,
            SourceSystemId,
            OfficeNumber,
            ROW_NUMBER() OVER (
                PARTITION BY SourceSystemId, OfficeNumber
                ORDER BY SourceSystemId, OfficeNumber
            ) AS rnb
        FROM {client}
        WHERE OfficeNumber <> 0
          AND SourceSystem = 'HCHB'
    ) a
    WHERE rnb = 1
) b
    ON CAST(b.SourceSystemId AS STRING) = CAST(cea.epi_id AS STRING)
    AND ABS(CAST(inv.BranchID AS INT)) = b.OfficeNumber

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
        WHEN ard.TotalServAdjustments = 0 THEN 'Payment'
    END

WHERE ard.grouping = 'BHHC'
    AND CAST(ard.loaddate AS DATE) = CURRENT_DATE()

UNION ALL

-- Second SELECT - CUBHUB System (BHHC grouping with E/F patterns)
SELECT
 
    CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS create_date_key,
    dt2.DateKey AS payment_date_key,
    dt3.DateKey AS statement_start_date_key,
    dt4.DateKey AS statement_end_date_key,
    s.SourceSystemKey AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    b.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.PayerICN AS payer_icn,
    ard.PayerName AS ability_payor_name,
    ard.PatientName AS patient_name,
    ard.MemberID AS member_id,
    ard.AdjustmentCodes AS adjustment_codes,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS  patientctlno,
    ard.PatientCtlNo AS invoice_number,
    CAST(ard.Charge AS DECIMAL(19,4)) AS charge,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS total_service_adjustments,
    CAST(ard.ClaimPayment AS DECIMAL(19,4)) AS claim_payment,
    CAST(ard.UnderPaymentAmount AS DECIMAL(19,4)) AS underpayment_amount,
    ard.Check_EFTNumber AS check_eftnumber,
    CURRENT_TIMESTAMP() AS Loaded_ts
FROM {abilityremittancedetails} ard

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey) AS rnb
        FROM {payerdimension}
        WHERE SourceSystemKey = 19
    ) a
    WHERE rnb = 1
) pd
    ON ard.PayerName = pd.Name

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'CUBHUB'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = CAST(ard.CreateDate AS DATE)

LEFT JOIN {date} dt2
    ON dt2.CalendarDate = CAST(ard.PaymentDate AS DATE)

LEFT JOIN {date} dt3
    ON dt3.CalendarDate = CAST(ard.StatementStart AS DATE)

LEFT JOIN {date} dt4
    ON dt4.CalendarDate = CAST(ard.StatementEnd AS DATE)

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            clientKey,
            ConformedLastName,
            ConformedFirstName,
            OfficeNumber,
            ROW_NUMBER() OVER (PARTITION BY ConformedLastName, ConformedFirstName ORDER BY ClientKey DESC) AS rnb
        FROM {client}
        WHERE OfficeNumber <> 0 AND SourceSystem = 'CUBHUB'
    ) a
    WHERE rnb = 1
) b
    ON ard.PatientName = CONCAT(b.ConformedLastName, ', ', b.ConformedFirstName)

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = b.OfficeNumber

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
        WHEN ard.TotalServAdjustments = 0 THEN 'Payment'
    END

WHERE ard.grouping = 'BHHC'
    AND CAST(ard.loaddate AS DATE) = CURRENT_DATE()
    AND (
        ard.PatientCtlNo RLIKE '^[0-9].*[0-9][EF][a-z][0-9].*[0-9]$$'
        OR ard.PatientCtlNo RLIKE '^[0-9].*[0-9][EF][a-z]S[0-9].*[0-9]$$'
        OR ard.PatientCtlNo RLIKE '^M-[0-9].*[0-9][EF][a-z][0-9].*[0-9]$$'
        OR ard.PatientCtlNo RLIKE '^M-[0-9].*[0-9][EF][a-z]S[0-9].*[0-9]$$'
    )

UNION ALL

-- Third SELECT - BEARS System (BHHC grouping, excludes both HCHB and CUBHUB patterns)
SELECT 

    CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS create_date_key,
    dt2.DateKey AS payment_date_key,
    dt3.DateKey AS statement_start_date_key,
    dt4.DateKey AS statement_end_date_key,
    CASE WHEN tmp2.INVNO IS NOT NULL THEN s.SourceSystemKey END AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    e.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.PayerICN AS payer_icn,
    ard.PayerName AS ability_payor_name,
    ard.PatientName AS patient_name,
    ard.MemberID AS member_id,
    ard.AdjustmentCodes AS adjustment_codes,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS  patientctlno,
    CASE WHEN tmp2.INVNO IS NOT NULL THEN RIGHT(ard.PatientCtlNo, 8) END AS invoice_number,
    CAST(ard.Charge AS DECIMAL(19,4)) AS charge,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS total_service_adjustments,
    CAST(ard.ClaimPayment AS DECIMAL(19,4)) AS claim_payment,
    CAST(ard.UnderPaymentAmount AS DECIMAL(19,4)) AS underpayment_amount,
    ard.Check_EFTNumber AS check_eftnumber,
    CURRENT_TIMESTAMP() AS Loaded_ts
FROM (
    -- Subquery to filter records not in tmp1 and not CUBHUB patterns
    SELECT 
        ard.*
    FROM {abilityremittancedetails} ard
    LEFT JOIN tmp1 inv
        ON CAST(ard.PatientCtlNo AS STRING) = CAST(inv.InvNum AS STRING)
        AND (ard.PatientCtlNo RLIKE '^2[89][90123].*' OR ard.PatientCtlNo RLIKE '^1.*')
    WHERE inv.InvNum IS NULL
        AND CAST(ard.loaddate AS DATE) = CURRENT_DATE()
        AND ard.grouping = 'BHHC'
        AND ard.PatientCtlNo NOT IN (
            SELECT DISTINCT PatientCtlNo
            FROM {abilityremittancedetails}
            WHERE (
                    PatientCtlNo RLIKE '^[0-9].*[0-9][EF][a-z][0-9].*[0-9]$$'
                    OR PatientCtlNo RLIKE '^[0-9].*[0-9][EF][a-z]S[0-9].*[0-9]$$'
                    OR PatientCtlNo RLIKE '^M-[0-9].*[0-9][EF][a-z][0-9].*[0-9]$$'
                    OR PatientCtlNo RLIKE '^M-[0-9].*[0-9][EF][a-z]S[0-9].*[0-9]$$'
                )
                AND CAST(ard.loaddate AS DATE) = CURRENT_DATE()
        )
) ard

LEFT JOIN tmp2
    ON tmp2.INVNO = RIGHT(ard.PatientCtlNo, 8) 
    AND LENGTH(ard.PatientCtlNo) <> 7

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = tmp2.office

LEFT JOIN {payerdimension} pd
    ON pd.PayerID = tmp2.BILLTO

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'BEARS'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = CAST(ard.CreateDate AS DATE)

LEFT JOIN {date} dt2
    ON dt2.CalendarDate = CAST(ard.PaymentDate AS DATE)

LEFT JOIN {date} dt3
    ON dt3.CalendarDate = CAST(ard.StatementStart AS DATE)

LEFT JOIN {date} dt4
    ON dt4.CalendarDate = CAST(ard.StatementEnd AS DATE)

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            sourcesystemid,
            officenumber,
            ClientKey,
            ROW_NUMBER() OVER (PARTITION BY sourcesystemid ORDER BY clientkey DESC) AS rnb
        FROM {client}
        WHERE officenumber <> 0 AND sourcesystem = 'BEARS'
    ) d
    WHERE rnb = 1
) e
    ON e.SourceSystemId = tmp2.CLIENTNO 
    AND e.OfficeNumber = tmp2.Office

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
        WHEN ard.TotalServAdjustments = 0 THEN 'Payment'
    END

UNION ALL

-- Fourth SELECT - AHC System
SELECT 

    CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS create_date_key,
    dt2.DateKey AS payment_date_key,
    dt3.DateKey AS statement_start_date_key,
    dt4.DateKey AS statement_end_date_key,
    s.SourceSystemKey AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    b.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.PayerICN AS payer_icn,
    ard.PayerName AS ability_payor_name,
    ard.PatientName AS patient_name,
    ard.MemberID AS member_id,
    ard.AdjustmentCodes AS adjustment_codes,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS  patientctlno,
    ard.PatientCtlNo AS invoice_number,
    CAST(ard.Charge AS DECIMAL(19,4)) AS charge,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS total_service_adjustments,
    CAST(ard.ClaimPayment AS DECIMAL(19,4)) AS claim_payment,
    CAST(ard.UnderPaymentAmount AS DECIMAL(19,4)) AS underpayment_amount,
    ard.Check_EFTNumber AS check_eftnumber,
    CURRENT_TIMESTAMP() AS Loaded_ts
FROM {abilityremittancedetails} ard

LEFT JOIN tmp1 inv
    ON CAST(ard.PatientCtlNo AS STRING) = CAST(inv.InvNum AS STRING)

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ABS(CAST(inv.BranchID AS INT))

LEFT JOIN {payerdimension} pd
    ON CAST(inv.GroupID AS STRING) = CAST(pd.PayerID AS STRING)

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'HCHB'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = CAST(ard.CreateDate AS DATE)

LEFT JOIN {date} dt2
    ON dt2.CalendarDate = CAST(ard.PaymentDate AS DATE)

LEFT JOIN {date} dt3
    ON dt3.CalendarDate = CAST(ard.StatementStart AS DATE)

LEFT JOIN {date} dt4
    ON dt4.CalendarDate = CAST(ard.StatementEnd AS DATE)

LEFT JOIN {client_episode_fs} cefs
    ON CAST(inv.PrimID AS BIGINT) = CAST(cefs.cefs_id AS BIGINT)

LEFT JOIN {client_episodes_all} cea
    ON cea.epi_id = cefs.cefs_epiid

LEFT JOIN (
    SELECT *
    FROM (
        SELECT 
            clientKey,
            SourceSystemId,
            OfficeNumber,
            ROW_NUMBER() OVER (
                PARTITION BY SourceSystemId, OfficeNumber
                ORDER BY SourceSystemId, OfficeNumber
            ) AS rnb
        FROM {client}
        WHERE OfficeNumber <> 0
          AND SourceSystem = 'HCHB'
    ) a
    WHERE rnb = 1
) b
    ON CAST(b.SourceSystemId AS STRING) = CAST(cea.epi_id AS STRING)
    AND ABS(CAST(inv.BranchID AS INT)) = b.OfficeNumber

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 
            THEN SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
        WHEN ard.TotalServAdjustments = 0 THEN 'Payment'
    END

WHERE ard.grouping = 'AHC'
  AND CAST(ard.loaddate AS DATE) = CURRENT_DATE()
""")

print("=" * 60)
print("FACT REMITTANCE DATA INSERTED SUCCESSFULLY")
print("=" * 60)